# 🏗️ Medallion Architecture Setup
This notebook automates the creation of **External Locations** and **Managed Schemas** for the Bronze, Silver, and Gold layers.

### 📋 Prerequisites
1. **Storage Credential:** `megaec-dev-credential` must be created in the Catalog UI.
2. **Key Vault:** The `storage-account-name` secret must be present in the `dev-scope`.

In [0]:
# COMMAND ----------
# Fetch storage account name from Key Vault
storage_account = dbutils.secrets.get(scope="dev-scope", key="storage-account-key")
credential_name = "databricks-analytics-dev-credential"



In [0]:
# 1. Define configuration
storage_account = "databrksanlytcdev"
credential_name = "databricks-analytics-dev-credential"
layers = ["bronze", "silver", "gold"]

# 2. Loop through layers and create locations
for layer in layers:
    location_name = f"{layer}_location"
    url = f"abfss://{layer}@{storage_account}.dfs.core.windows.net/"
    
    print(f"Creating External Location: {location_name}...")
    
    # We use f-strings to inject our variables into the SQL command
    # Added 'SKIP ASSET CREATION' to avoid the 403 File Events error we saw earlier
    spark.sql(f"""
        CREATE EXTERNAL LOCATION IF NOT EXISTS `{location_name}`
        URL '{url}'
        WITH (STORAGE CREDENTIAL `{credential_name}`)
         
    """)

print("\n✅ All external locations processed. Verification:")
display(spark.sql("SHOW EXTERNAL LOCATIONS"))

In [0]:
catalog_name = "dev_catalog"
# ── Catalog and Schemas ───────────────────────────────────────────────────────
spark.sql(f""" CREATE CATALOG IF NOT EXISTS `{catalog_name}` """)
spark.sql(f"""
    USE CATALOG `{catalog_name}`
""")

# Create schemas for each layer)
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")


In [0]:
print("Unity Catalog setup complete")
spark.sql(f""" SHOW SCHEMAS IN `{catalog_name}` """).show()